In [ ]:
# --- Requirements ---
!pip install -U langchain langchain-community  sentence-transformers datasets scikit-learn numpy

In [2]:
import os
import shutil
import numpy as np
from datasets import load_dataset
from sklearn.metrics import ndcg_score
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

In [3]:
# ----------------------------
# 1) Load Dataset
# ----------------------------
print("Loading MedQA...")
ds = load_dataset("medalpaca/medical_meadow_medqa", split="train")

Loading MedQA...


In [4]:
# ----------------------------
# 2) Create Documents
# ----------------------------
docs = []
for i, item in enumerate(ds):
    q = item["input"].strip()
    a = item["output"].strip()
    text = f"Medical Case: {q}\nDiagnosis/Action: {a}"
    docs.append(Document(page_content=text, metadata={"id": str(i)}))

print(f"Number of documents: {len(docs)}")

Number of documents: 10178


In [5]:
# ----------------------------
# 3) Build Vectorstore
# ----------------------------
db_dir = "./vector_db_medqa"
if os.path.exists(db_dir):
    shutil.rmtree(db_dir)

# Strong embeddings with normalization
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},  # use "cpu" if no GPU
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma.from_documents(docs, embeddings, persist_directory=db_dir)
print("Vectorstore created successfully.")

Vectorstore created successfully.


In [6]:
# ----------------------------
# 4) Two-Stage Retriever Setup
# ----------------------------
# Stage 1: Bi-Encoder retrieval
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# Stage 2: Cross-Encoder reranking
rerank_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=rerank_model, top_n=5)

# High-performance retriever
improved_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [7]:
# ----------------------------
# 5) Normalization and Metrics
# ----------------------------
def normalize(text: str):
    return " ".join(text.lower().strip().split())

def calculate_metrics(gt_dict, k=5):
    mrr = 0.0
    recall = 0
    ndcg_list = []

    for q, answers in gt_dict.items():
        retrieved_docs = improved_retriever.invoke(q)
        relevance = []
        found_hit = False

        for rank, doc in enumerate(retrieved_docs, start=1):
            doc_text = normalize(doc.page_content)
            is_match = any(normalize(ans) in doc_text for ans in answers)
            rel_score = 1 if is_match else 0
            relevance.append(rel_score)

            if is_match and not found_hit:
                mrr += 1.0 / rank
                recall += 1
                found_hit = True

        # Pad for NDCG
        if len(relevance) < k:
            relevance.extend([0] * (k - len(relevance)))

        actual = np.array([relevance[:k]])
        ideal = np.array([[1] + [0]*(k-1)])  # ideal: correct at top
        ndcg_list.append(ndcg_score(ideal, actual) if np.sum(actual) > 0 else 0.0)

    return {
        "MRR": mrr / len(gt_dict),
        "Recall@k": recall / len(gt_dict),
        "nDCG@k": np.mean(ndcg_list)
    }

In [8]:
# ----------------------------
# 6) Prepare 20 Reworded Questions
# ----------------------------
ground_truth = {
    "A pregnant woman at 22 weeks experiences burning urination that worsens despite home remedies. Which antibiotic is most suitable?": ["Nitrofurantoin"],
    "An infant dies suddenly in sleep; what prevention measure reduces risk of SIDS?": ["Supine sleep on a firm mattress"],
    "A 3-week-old with persistent yellow vomiting and fussiness; what developmental migration is abnormal?": ["Abnormal migration of ventral pancreatic bud"],
    "A woman post-femur surgery has sudden hypoxic respiratory failure; pulmonary autopsy shows fibrous tissue in arteries. Likely cause?": ["Thromboembolism"],
    "A woman with lifelong heavy periods, easy bruising, normal PT, elevated PTT; most likely condition?": ["Von Willebrand disease"],
    "A zookeeper with severe abdominal pain radiating to back and nausea; likely cause?": ["Scorpion sting"],
    "A third-trimester pregnant woman with hypertension and edema; which test confirms proteinuria?": ["24-hour urine protein"],
    "A term infant with pleuroperitoneal membrane defect on ultrasound; diagnosis?": ["Gastric fundus in the thorax"],
    "A chronic smoker with productive cough and exertional dyspnea; airflow abnormality?": ["Progressive obstruction of expiratory airflow"],
    "A woman with diplopia, ocular pain, and thyroid enlargement; orbital change cause?": ["Glycosaminoglycan accumulation in the orbit"],
    "A toddler with progressive weakness, ptosis, weak gag reflex, shallow breathing; mechanism?": ["Blockade of presynaptic acetylcholine release at the neuromuscular junction"],
    "An infant with seizures, developmental delay, microcephaly, light pigmentation, musty odor; genetic principle?": ["Pleiotropy"],
    "A young man with hearing loss and cerebellopontine angle mass; embryologic origin?": ["Neural crest"],
    "A senior man with sudden leg pain and pale extremity; immediate intervention?": ["Heparin drip"],
    "A man with ureter stone and hydronephrosis; urinary pH?": ["Urinary pH: 4.7"],
    "Adult with influenza; which drug mechanism is best?": ["Inhibition of neuraminidase"],
    "A woman with stab wound to neck and stable vitals; next imaging?": ["CT angiogram"],
    "A teenage girl with neck lumps that move on swallowing; likely cause of pregnancy loss?": ["Chromosomal abnormalities"],
    "An infant with distended abdomen and painful crying; developmental failure diagnosis?": ["Hirschsprung disease"],
    "A chronic smoker with leg pain and hair loss; vascular pathology?": ["Narrowing and calcification of vessels"]
}

In [9]:
# ----------------------------
# 7) Run Evaluation
# ----------------------------
print("\nStarting Advanced Evaluation...")
results = calculate_metrics(ground_truth, k=5)

print(f"\n--- Metrics Summary ---")
for m, v in results.items():
    print(f"{m}: {v:.4f}")


Starting Advanced Evaluation...

--- Metrics Summary ---
MRR: 0.6375
Recall@k: 0.7500
nDCG@k: 0.6290


In [10]:
import shutil

# Replace "vector_db" with your folder path
shutil.make_archive("vector_db_zip", 'zip', "vector_db_medqa")

'/content/vector_db_zip.zip'

In [11]:
from google.colab import files

files.download("vector_db_zip.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>